In [2]:
import dspy
import json
import random
import logging
from pathlib import Path
from typing import Dict, List, Any
from dotenv import load_dotenv
from tqdm import tqdm
from itertools import combinations
import sys

# --- Add project root to path ---
project_root = str(Path.cwd().parent) 
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- Load Config & Environment ---
try:
    from config import settings
    from src.data_processing.utils import load_corpus
    from src.rag_system.data_stores import PPLDataStores
except ImportError as e:
    print(f"Error importing project modules: {e}")
    print("Ensure you have run 'pip install -e .' in the project root and the structure is correct.")

load_dotenv()

# --- Configure Logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("MultiHopGenerator")

# --- Configure LLMs ---
# This pipeline requires powerful reasoning. We'll use the "teacher" model for all steps.
try:
    # Use the powerful model you've defined for compiling/teaching
    teacher_lm = dspy.LM(settings.TEACHER_LM_MODEL, **settings.TEACHER_LM_KWARGS)
    dspy.configure(lm=teacher_lm)
    logger.info(f"✅ DSPy LM configured with: {settings.TEACHER_LM_MODEL}")
except Exception as e:
    logger.error(f"❌ Failed to configure LLM: {e}", exc_info=True)
    # Stop execution if we can't configure the LM
    raise

# --- Load Data Stores & Corpus ---
try:
    stores = PPLDataStores(settings.as_dict())
    metastore = stores.metadata_vectorstore
    logger.info("✅ Metadata vector store loaded.")
    
    all_documents = load_corpus(settings.CORPUS_FILE)
    # Create the lookup map (title -> full doc dict)
    doc_map = {doc['title'].lower(): doc for doc in all_documents if 'title' in doc}
    logger.info(f"✅ Loaded {len(all_documents)} documents and created title map.")
except Exception as e:
    logger.error(f"❌ Failed to load data stores or corpus: {e}", exc_info=True)
    raise

2025-10-23 04:17:09,977 - INFO - ✅ DSPy LM configured with: anthropic/claude-sonnet-4-20250514
2025-10-23 04:17:10,000 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


--- Initializing PPL Data Stores ---
  -> Loading collection 'PPL_agentic_documents' from /home/webui/Project/data/stores/chroma_agentic_db...


2025-10-23 04:17:10,190 - INFO - Use pytorch device_name: cpu
2025-10-23 04:17:10,193 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2
2025-10-23 04:17:13,329 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2025-10-23 04:17:13,352 - INFO - Use pytorch device_name: cpu
2025-10-23 04:17:13,353 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2


  -> Loading collection 'PPL_text_split_documents' from /home/webui/Project/data/stores/chroma_text_split_db...


2025-10-23 04:17:16,251 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2025-10-23 04:17:16,273 - INFO - Use pytorch device_name: cpu
2025-10-23 04:17:16,275 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2


  -> Loading collection 'PPL_document_metadata' from /home/webui/Project/data/stores/chroma_metadata_db...
  -> Loading BM25 retriever from /home/webui/Project/data/stores/bm25_agentic_retriever.pkl...
  -> Loading BM25 retriever from /home/webui/Project/data/stores/bm25_textsplit_retriever.pkl...


2025-10-23 04:17:20,354 - INFO - ✅ Metadata vector store loaded.
2025-10-23 04:17:20,394 - INFO - ✅ Loaded 419 documents and created title map.


✅ All data stores loaded successfully.


In [3]:
class GenerateFollowUpTopicsOld(dspy.Signature):
    """
    Analyzes a policy document snippet to identify **at most 3** synergistic document
    titles that fill informational gaps, forming a coherent multi-hop cluster.
    """
    
    __doc__ = """
    You are a research expert and systems architect analyzing the UQ Policy and Procedure Library (PPL). Your primary mission is to identify **at most 3** other document titles that are **synergistic** with the provided document context.

    A "synergistic" document is one that fills a critical "informational gap." A user would need to read **both** the source document **and** the synergistic document(s) to understand a complete process or answer a complex, multi-faceted question.

    **Instructions:**
    1.  **Analyze the Context:** Read the `document_context` (title, description, snippet).
    2.  **Find Informational Gaps:** Actively look for processes, rules, or terms that are mentioned but **not fully explained**.
    3.  **Select for Synergy:** Select **at most 3** titles that form the most coherent and necessary *cluster* for a complex user query.
    4.  **CRITICAL:** If you cannot find any strongly synergistic, necessary, and related document titles, **return an empty list.** Do not invent weak connections.
    5.  **Output Titles Only:** Generate *only* the most likely, exact document titles.

    ---
    **Example 1: Strong Synergy (Process Flow)**
    Input Context: ...Staff must adhere to the [Information Management Policy](URL1) ... report any suspected breaches according to the [Cyber Security Incident Response Procedure](URL2). ... subject to disciplinary action, as detailed in the 'Staff Grievance Resolution Procedure'.
    Example Output:
    [
      "Information Management Policy",
      "Cyber Security Incident Response Procedure",
      "Staff Grievance Resolution Procedure"
    ]

    ---
    **Example 2: Strong Synergy (Detailed Procedure)**
    Input Context: ...For spills involving highly toxic or specialized substances, such as arsenic compounds, consult the specific substance safety guideline...
    Example Output:
    [
      "Working Safely with Arsenic Guideline",
      "Chemical Spill and Response Procedure"
    ]

    ---
    **Example 3: No Strong Synergy**
    Input Context:
    Title: UQ Travel Policy
    Description: Rules for booking flights and accommodation.
    Snippet: ...All travel must be booked through the UQ-approved travel provider. Reimbursements are handled via the finance portal...
    Example Output:
    []
    *(Reasoning: While related, "Finance Reimbursement Procedure" isn't *necessary* to understand the *travel booking* process. The link is weak.)*
    """
    document_context: str = dspy.InputField(desc="The title, description, and a text snippet of a single policy document.")
    related_document_titles: List[str] = dspy.OutputField(desc="A Python list of **at most 3** synergistic document titles. Returns an empty list if no strong synergy is found.")


class GenerateFollowUpTopics(dspy.Signature):
    """
    Analyzes a policy document snippet to **extract** a synergistic cluster of at 
    most 3 **explicitly mentioned**, related document titles.
    """
    
    __doc__ = """
    You are a research expert and systems architect analyzing the UQ Policy and Procedure Library (PPL). Your primary mission is to identify **at most 3** other document titles that are **synergistic** with the provided document context.

    A "synergistic" document is one that fills a critical "informational gap." A user would need to read **both** the source document **and** the synergistic document(s) to understand a complete process or answer a complex, multi-faceted question.

    **Instructions:**
    1.  **CRITICAL: EXTRACT, DO NOT INVENT.** Your *only* task is to find document titles that are **explicitly mentioned or linked to** in the `document_context`. Do NOT infer or invent titles that are not written in the text.
    2.  **Analyze the Context:** Read the `document_context` (title, description, snippet). Identify key terms, mentioned procedures, cross-references, or compliance requirements that are *also* explicitly named as other documents.
    3.  **Find Informational Gaps:** From the titles you extracted, identify which ones fill a critical "informational gap" (e.g., they are the next step in a process).
    4.  **Select for Synergy:** From your *extracted candidate list*, select **at most 3** titles that form the most coherent and necessary *cluster* for a complex user query.
    5.  **Return Empty List:** If you cannot find any *explicitly mentioned* titles, OR if the ones you find are not strongly synergistic, **return an empty list.**
    6.  **Output Titles Only:** Generate *only* the most likely, exact document titles.

    ---
    **Example 1: Strong Synergy (Process Flow)**
    Input Context: ...Staff must adhere to the [Information Management Policy](URL1) ... report any suspected breaches according to the [Cyber Security Incident Response Procedure](URL2). ... subject to disciplinary action, as detailed in the 'Staff Grievance Resolution Procedure'.
    Example Output:
    [
      "Information Management Policy",
      "Cyber Security Incident Response Procedure",
      "Staff Grievance Resolution Procedure"
    ]

    ---
    **Example 2: Strong Synergy (Detailed Procedure)**
    Input Context: ...For spills involving highly toxic arsenic-based substances, such as arsenic trioxide, consult the [Working Safely with Arsenic Guideline](URL1) for specific instructions on how to...
    Example Output:
    [
      "Working Safely with Arsenic Guideline",
      "Chemical Spill and Response Procedure"
    ]
    *(Reasoning: The snippet explicitly names the "Working Safely with Arsenic Guideline" document which fills an information gap about a specific topic.)*

    ---
    **Example 3: No Strong Synergy / No Explicit Title**
    Input Context:
    Title: UQ Travel Policy
    Description: Rules for booking flights and accommodation.
    Snippet: ...All travel must be booked through the UQ-approved travel provider. Reimbursements are handled via the finance portal...
    Example Output:
    []
    *(Reasoning: The text mentions a "finance portal" but does **not** explicitly state a document title like "Finance Payment Policy". Therefore, we don't add it to the final list.)*
    """
    document_context: str = dspy.InputField(desc="The title, description, and a text snippet of a single policy document.")
    related_document_titles: List[str] = dspy.OutputField(desc="A Python list of **at most 3** synergistic document titles **explicitly found** in the text. Returns an empty list if none are found.")

class GenerateMultiHopQA(dspy.Signature):
    """
    Given the full text of MULTIPLE related university policy documents, your task is to generate one complex, targeted question that REQUIRES synthesizing information from ALL provided documents to answer. Then, provide the comprehensive answer.
    """

    __doc__ = """
    You are an expert curriculum designer creating a difficult test question for a university policy exam. You are given the full text of 2-3 related policy documents. Your task is to generate one complex question that a student could **only** answer by finding and synthesizing specific facts from **ALL** of the provided documents.

    ---
    **Crucial Guidelines for the Question:**
    - **Focus on Synthesis:** The question *must* create a scenario that explicitly links concepts or processes from each document.
    - **Require ALL Documents:** It must be impossible to answer the question fully using only one of the documents.
    - **The goal is maximum logical complexity and synthesis, NOT realism.** The question should be specific, targeted, and fact-based. Avoid vague or generic scenarios.
    - **Be specific:** Refer to processes, terms, or conditions from the documents to create the connection.
    ---

    The answer must be detailed, authoritative, and explicitly reference and synthesize facts from **all provided documents** to prove the connection.
    """
    # We remove the 'persona' field as it encourages vague, "realistic" scenarios
    documents_context: str = dspy.InputField(desc="The concatenated full text of multiple distinct policy documents.")
    question: str = dspy.OutputField(desc="A complex, multi-hop question requiring information from all documents.")
    answer: str = dspy.OutputField(desc="A detailed, gold-standard answer synthesizing information from all documents.")

class ValidateMultiHopQA(dspy.Signature):
    """
    Given a question, a generated answer, and a set of source documents,
    validate the quality of the QA pair.
    """
    
    __doc__ = """
    You are a meticulous Quality Assurance agent. Your task is to validate a synthetically generated Question-Answer pair based on a set of source documents.
    You must check three criteria:

    1.  **Faithfulness:** Is the `generated_answer` 100% factually supported by the information *only* within the provided `document_contexts`? (No external knowledge).
    2.  **Necessity:** Does answering the `question` *truly require* synthesizing information from **ALL** of the provided `document_titles`? (i.e., it couldn't be answered by just one).
    3.  **Clarity:** Is the `question` specific, clear, and unambiguous?

    Respond with a JSON object with two keys:
    - "is_valid": true or false. `true` only if ALL three criteria are met.
    - "reasoning": A brief explanation for your decision, especially if invalid.
    """
    question = dspy.InputField(desc="The generated multi-hop question.")
    generated_answer = dspy.InputField(desc="The generated answer to the question.")
    document_titles = dspy.InputField(desc="A list of the source document titles.")
    document_contexts = dspy.InputField(desc="The full text of the source documents.")
    
    validation_json = dspy.OutputField(desc="A JSON object with 'is_valid' (bool) and 'reasoning' (str) keys.")

# Instantiate the modules
topic_generator = dspy.ChainOfThought(GenerateFollowUpTopics)
qa_generator = dspy.ChainOfThought(GenerateMultiHopQA)
validator = dspy.ChainOfThought(ValidateMultiHopQA)

logger.info("✅ All generation and validation agents initialized.")

2025-10-23 04:17:20,460 - INFO - ✅ All generation and validation agents initialized.


In [5]:
# --- Configuration ---
NUM_SEED_DOCUMENTS_TO_TRY = 5  # How many random docs to start with
TEST_SET_SPLIT_RATIO = 0.3 # 30% of new examples go to the test set

logger.info(f"--- Starting automated multi-hop generation pipeline for {NUM_SEED_DOCUMENTS_TO_TRY} seed documents ---")

new_examples_generated = 0

# --- Initialize lists *before* the main loop ---
test_data = []
train_data = []

# --- Main Generation Loop ---
seed_documents = random.sample(all_documents, min(NUM_SEED_DOCUMENTS_TO_TRY, len(all_documents)))

for seed_doc in seed_documents:
    try:
        print("processing seed: ", seed_doc['title'])

        # --- Step 1: Generate Candidate Topics ---
        seed_context = (f"Title: {seed_doc['title']}\n"
                        f"Description: {seed_doc.get('description', '')}\n"
                        f"Snippet: {seed_doc.get('text', 'NO SNIPPET FOUND')}")
        
        topic_pred = topic_generator(document_context=seed_context)

        # --- Step 2: Create Super-Cluster and Subsets ---
        related_titles = topic_pred.related_document_titles 
        
        if not related_titles:
            continue # No related titles found for this seed

        full_cluster_titles = set([seed_doc['title']])
        full_cluster_titles.update(related_titles)

        subsets_to_process = []
        if len(full_cluster_titles) >= 2:
            subsets_to_process.extend(list(combinations(full_cluster_titles, 2)))
        if len(full_cluster_titles) >= 3:
            subsets_to_process.extend(list(combinations(full_cluster_titles, 3)))
        
        if not subsets_to_process:
            continue

        # --- Step 3: Iterate Subsets and Generate QA ---
        for doc_group_titles in tqdm(subsets_to_process, total=len(subsets_to_process), desc="processing subsets"):
            try:
                # --- Step 3a: Validate subset titles & get docs ---
                doc_group = []
                valid_group = True
                for title in doc_group_titles:
                    doc = doc_map.get(title.lower())
                    if doc:
                        doc_group.append(doc)
                    else:
                        logger.warning(f"Title '{title}' from LLM-generated group not in doc_map. Skipping subset.")
                        valid_group = False
                        break
                
                if not valid_group:
                    continue

                # --- Step 3b: Generate QA pair ---
                doc_contexts = [d['text'] for d in doc_group]
                context_for_qa = "\n\n--- NEW DOCUMENT ---\n\n".join(doc_contexts)
                
                # Removed 'persona' argument
                qa_pred = qa_generator(documents_context=context_for_qa)
                
                if qa_pred.question.strip().upper() == "NO SCENARIO POSSIBLE":
                    logger.info(f"Skipping subset {doc_group_titles}, agent returned 'NO SCENARIO POSSIBLE'")
                    continue

                # --- Step 3c: Validate the Generated QA Pair ---
                val_pred = validator(
                    question=qa_pred.question, # Use the correct field name
                    generated_answer=qa_pred.answer,
                    document_titles=list(doc_group_titles),
                    document_contexts=context_for_qa
                )    
                
                validation_output = json.loads(val_pred.validation_json)
                
                if validation_output.get("is_valid", False):
                    new_examples_generated += 1
                    
                    # Create the final example dict (distractors removed)
                    new_example = {
                        "question": qa_pred.question,
                        "gold_answer": qa_pred.answer,
                        "gold_titles": list(doc_group_titles),
                        "hop_count": len(doc_group_titles),
                    }
                    
                    # Add to train or test set
                    if random.random() < TEST_SET_SPLIT_RATIO:
                        test_data.append(new_example)
                    else:
                        train_data.append(new_example)
                
                else:
                    logger.warning(f"Rejected QA pair for group {doc_group_titles}. Reason: {validation_output.get('reasoning')}")
                    pass

            except Exception as e:
                logger.warning(f"Error in inner loop for {doc_group_titles}: {e}", exc_info=True)
                continue

    except Exception as e:
        logger.error(f"Error in outer loop for seed {seed_doc['title']}: {e}", exc_info=True)
        continue

# --- Step 5: Save Final, Appended Datasets ---
logger.info(f"Pipeline finished. Generated {new_examples_generated} new validated multi-hop examples.")
logger.info("Appending new examples to train.jsonl and test.jsonl...")

# Appending to files in the current notebook directory
try:
    with open("train.jsonl", "a", encoding="utf-8") as f:
        for item in train_data:
            f.write(json.dumps(item) + '\n')
    logger.info(f"Appended {len(train_data)} examples to train.jsonl")

    with open("test.jsonl", "a", encoding="utf-8") as f:
        for item in test_data:
            f.write(json.dumps(item) + '\n')
    logger.info(f"Appended {len(test_data)} examples to test.jsonl")

except Exception as e:
    logger.error(f"Error saving files: {e}", exc_info=True)

logger.info("--- All done! ---")

2025-10-23 04:21:20,609 - INFO - --- Starting automated multi-hop generation pipeline for 5 seed documents ---


processing seed:  Chemical Manifest Procedure


processing subsets: 100%|██████████| 10/10 [08:09<00:00, 48.97s/it]


processing seed:  Animal Ethics in Teaching and Research Procedure


processing subsets: 100%|██████████| 10/10 [07:26<00:00, 44.70s/it]


...

processing subsets: 100%|██████████| 4/4 [02:39<00:00, 39.99s/it]


processing seed:  Chemical Storage Safety Guideline


processing subsets:  10%|█         | 1/10 [00:49<07:25, 49.47s/it]2025/10/23 04:41:26 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025-10-23 04:41:31,459 - WARNING - Error in inner loop for ('Storage of Chemicals in Fridges, Freezers and Cold Rooms Guideline', 'Safety Data Sheets Guideline'): litellm.RateLimitError: AnthropicException - {"type":"error","error":{"type":"rate_limit_error","message":"This request would exceed the rate limit for your organization (ee272d87-501a-4f52-ab47-19200d544d7b) of 30,000 input tokens per minute. For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://www.anthropic.com/contact-sales to discuss your options for a rate limit increase."},"request_id":"req_011CUPWuAVNCbfkNjqU2BDDD"}
Traceback (most recent call las

processing seed:  Student Identification Cards Procedure


processing subsets: 100%|██████████| 4/4 [03:01<00:00, 45.33s/it]
2025-10-23 04:50:34,293 - INFO - Pipeline finished. Generated 30 new validated multi-hop examples.
2025-10-23 04:50:34,295 - INFO - Appending new examples to train.jsonl and test.jsonl...
2025-10-23 04:50:34,299 - INFO - Appended 25 examples to train.jsonl
2025-10-23 04:50:34,302 - INFO - Appended 5 examples to test.jsonl
2025-10-23 04:50:34,304 - INFO - --- All done! ---


In [ ]:
random